sudo docker build -t jlr-doc-intel .

sudo docker run --rm --gpus all -p 8000:8000 -e OUTPUT_DIR=/outputs -e OPENAI_API_KEY=sXXXXXXXXX -v $(pwd)/outputs:/outputs -v ~/paddle_models:/root/.paddleocr -v $(pwd)/app:/app/app jlr-doc-intel

In [ ]:
import requests

url = "http://localhost:8000/doc-intel"

payload = {
    "document_url": "https://drive.google.com/uc?export=download&id=1zffFZhQeN_Q8AheoPKePZWZOr5YLnC7j",
    "request_id": "test123",
    "num_pages": 5,
    "model_type": "v3",
    "enable_ai_tables": True,
    "do_summary": True
}

response = requests.post(url, json=payload)

print(response.status_code)
print(response.text)

In [1]:
import fitz  # PyMuPDF
import pandas as pd
import cv2
import numpy as np
import re
import matplotlib.pyplot as plt

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Try imports to handle environment differences
try:
    from paddleocr import PPStructureV3, PaddleOCRVL
except ImportError:
    print("Warning: PaddleOCR not installed or specific models missing.")

def load_model(model_type="v3"):
    """
    Factory function to load the specific Paddle model.
    model_type: "v3" (PPStructureV3) or "vl" (PaddleOCRVL)
    """
    if model_type == "vl":
        print("Loading PaddleOCR VL Model...")
        # VL model usually handles orientation internally or doesn't expose the flag
        return PaddleOCRVL()
    else:
        print("Loading PaddleOCR PPStructureV3...")
        return PPStructureV3(
            layout_detection_model_name="PP-DocLayout-L", 
            text_recognition_model_name="en_PP-OCRv4_mobile_rec",
            device="gpu:0", 
            use_table_recognition=True,    
            use_doc_orientation_classify=False,
            use_region_detection=True,
            use_doc_unwarping=False
        )

def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union (IoU) for label matching (V3 specific)."""
    if not boxA or not boxB: return 0
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = float(boxAArea + boxBArea - interArea)
    return interArea / unionArea if unionArea != 0 else 0

def get_sorting_key(item):
    """
    Sorts blocks: Headers first, then Body (by AI order), then Footers.
    Robust against NoneTypes.
    """
    label = item.get('label', '').lower()
    
    # Priority Overrides
    if 'header' in label: return -1000
    if 'footer' in label: return 10000
    
    # Check for 'order' key (V3) or fallback to Y-coordinate (VL)
    if 'order' in item and item['order'] is not None:
        return int(item['order'])
    
    # Fallback for VL model or missing order: Sort by Y coordinate
    bbox = item.get('bbox', [0, 0, 0, 0])
    return int(bbox[1])

def get_compliance_patterns():
    return {
        # 1. TOC Pattern
        'toc_start_heading': r'(?i)^\s*(TABLE\s+OF\s+CONTENTS|CONTENTS)\s*$',
        
        # 2. High-Level Hierarchy (New: Chapter & General Section)
        'chapter_heading': r'^\s*(?i:CHAPTER)\s+([A-Z0-9\-\.\s]+).*',
        'section_general_heading': r'^\s*(?i:SECTION)\s+([A-Z0-9\-\.\s]+).*',
        
        # 3. Standard Regulatory Patterns
        'cfr_part_heading': r'^\s*((?:49\s+CFR\s+)?PART\s+\d+).*',
        'cfr_section_heading': r'^\s*(§\s*\d+(\.\d+)?)\s+(.*)',
        'fmvss_paragraph': r'^\s*(S\d+(\.\d+)*)\s+(.*)',
        'article_heading': r'^\s*((?i:ARTICLE))\s+(\d+)\b\.?\s*(?:[A-Z0-9"“\'].*)?$',
        'appendix_heading': r'^\s*(APPENDIX|ANNEX)\s+([A-Z0-9]+)\s*(.*)',
        'roman_upper_heading': r'^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV)\.\s+(?![A-Z0-9]\.)(.*)',
        
        # 4. Numbered Paragraphs
        'multi_level_heading': r'^\s*(?!49\s+CFR)["“\']?(\d{1,3}(?!\d)(\.\d+)*)\.?\s*(?:[A-Z0-9"“\'].*)?$',
        'numeric_paren_heading': r'^\s*\((\d+)\)\s*(?![\d]{3}[-\s–\u2013\u2014])(.*)',
        'alpha_paren_heading': r'^\s*\(([a-z])\)\s+(.*)',
    }

def hybrid_extract_data(pdf_path, pages_to_process, model, model_type="v3", exclude_labels=None):
    if exclude_labels is None:
        exclude_labels = ['header', 'footer']

    print(f"--- Processing {len(pages_to_process)} Pages (Model: {model_type.upper()}) ---")
    
    doc = fitz.open(pdf_path)
    
    # --- GLOBAL STATE ---
    all_chunks = []
    parent_stack = [(0, "ROOT")] 
    current_annex_context = "NIL"
    chunk_counter = 1
    in_toc_mode = False
    
    current_chunk = {
        'chunk_id': 1, 'clause_id': "NIL", 'parent_id': "ROOT", 'level': 0,
        'appendix/annex': "NIL", 'content_verbatim': "", 'source_page': ""
    }
    
    debug_images = {}
    
    patterns = get_compliance_patterns()
    regex_map = {k: re.compile(v) for k, v in patterns.items()}
    
    # Priority List
    heading_priority = [
        'toc_start_heading',
        'appendix_heading', 'chapter_heading', 'section_general_heading',
        'cfr_part_heading', 'cfr_section_heading', 'fmvss_paragraph', 'article_heading', 
        'multi_level_heading', 'roman_upper_heading', 'numeric_paren_heading', 'alpha_paren_heading'
    ]
    
    month_pattern = re.compile(r'(?i)\b(January|February|March|April|May|June|July|August|September|October|November|December)\b')

    for page_num in pages_to_process:
        if page_num >= len(doc): continue
        print(f"Processing Page {page_num}...")
        page = doc[page_num]
        
        # 1. VISUAL EXTRACTION
        zoom = 2.0 if model_type == "vl" else 1.5 
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)
        
        img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
        if pix.n == 4: img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2RGB)
        img_bgr = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)

        layout_results = model.predict(img_bgr)
        all_blocks = _parse_paddle_output(layout_results, model_type)
        
        valid_blocks = [b for b in all_blocks if b['label'].lower() not in exclude_labels]
        valid_blocks.sort(key=get_sorting_key)

        image_h, image_w = img_bgr.shape[:2]
        scale_x = page.rect.width / image_w
        scale_y = page.rect.height / image_h

        viz_image = img_bgr.copy()
        extracted_lines = [] 
        table_counter = 1
        
        if str(page_num) not in current_chunk['source_page']:
            if current_chunk['source_page']: current_chunk['source_page'] += f", {page_num}"
            else: current_chunk['source_page'] = str(page_num)

        for idx, block in enumerate(valid_blocks):
            label = block['label'].lower()
            bbox = block['bbox']
            
            x1, y1, x2, y2 = map(int, bbox)
            color = (0, 0, 255) if 'table' in label else (0, 255, 0)
            if 'title' in label: color = (255, 0, 0)
            
            cv2.rectangle(viz_image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(viz_image, f"[{idx+1}] {label.upper()}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            clean_text = ""
            if 'table' in label:
                clean_text = f"<Table_{table_counter}_Pg{page_num}>"
                extracted_lines.append((clean_text, label))
                table_counter += 1
            else:
                rect = fitz.Rect(bbox[0] * scale_x, bbox[1] * scale_y, bbox[2] * scale_x, bbox[3] * scale_y)
                text = page.get_text("text", clip=rect).strip()
                if text:
                    lines = text.split('\n')
                    extracted_lines.extend([(l.strip(), label) for l in lines if l.strip()])
        
        debug_images[page_num] = viz_image

        # 2. LOGIC PARSING
        for line, visual_label in extracted_lines:
            if line.startswith("<Table_"):
                current_chunk['content_verbatim'] += f"\n{line}\n"
                continue

            match_found = False; found_type = None; match_obj = None
            for h_type in heading_priority:
                m = regex_map[h_type].match(line)
                if m:
                    match_found = True; found_type = h_type; match_obj = m
                    break
            
            # --- TOC MODE ---
            if found_type == 'toc_start_heading':
                if current_chunk['content_verbatim'].strip(): 
                    all_chunks.append(current_chunk)
                    chunk_counter += 1
                in_toc_mode = True
                current_chunk = {
                    'chunk_id': chunk_counter, 'clause_id': "TOC", 'parent_id': "ROOT",
                    'level': 0, 'appendix/annex': "NIL", 
                    'content_verbatim': line + "\n", 'source_page': str(page_num)
                }
                continue 
            
            if in_toc_mode:
                if visual_label in ['content', 'list', 'text', 'table']:
                    current_chunk['content_verbatim'] += line + "\n"
                    continue
                is_toc_item = re.search(r'(\.{3,}|\s\d+)$', line)
                if is_toc_item:
                    current_chunk['content_verbatim'] += line + "\n"
                    continue
                if match_found:
                    in_toc_mode = False
                else:
                    current_chunk['content_verbatim'] += line + "\n"
                    continue

            # --- STANDARD PROCESSING ---
            if match_found:
                # 2a. EXTRACT ID & CALCULATE LEVEL
                level = 1
                
                # --- FIXED ID EXTRACTION LOGIC ---
                if found_type == 'appendix_heading':
                    raw_id = f"{match_obj.group(1)} {match_obj.group(2)}"
                elif found_type == 'article_heading':
                    # Pattern has 2 groups: ((?i:ARTICLE)) (\d+)
                    raw_id = f"{match_obj.group(1).upper()} {match_obj.group(2)}"
                elif found_type == 'chapter_heading':
                    # Pattern has 1 group: ([A-Z0-9...]+)
                    raw_id = f"CHAPTER {match_obj.group(1).strip()}"
                elif found_type == 'section_general_heading':
                    # Pattern has 1 group: ([A-Z0-9...]+)
                    raw_id = f"SECTION {match_obj.group(1).strip()}"
                elif found_type == 'cfr_part_heading':
                    raw_id = match_obj.group(1).strip().strip('.')
                else:
                    # Standard Paragraphs
                    raw_id = match_obj.group(1).strip().strip('.')
                    # Shift Numbered Paragraphs to Level 2
                    level = raw_id.count('.') + 2
                
                # --- GUARDS ---
                is_digit_only = raw_id.isdigit()
                if is_digit_only and len(raw_id) == 4 and (1900 <= int(raw_id) <= 2050):
                    current_chunk['content_verbatim'] += line + " "
                    continue
                if is_digit_only and len(raw_id) <= 2:
                    if month_pattern.search(line):
                        current_chunk['content_verbatim'] += line + " "
                        continue

                # --- CHUNK CREATION ---
                if current_chunk['content_verbatim'].strip(): 
                    all_chunks.append(current_chunk)
                    chunk_counter += 1
                
                # Level Logic
                if found_type in ['chapter_heading', 'section_general_heading', 'cfr_part_heading', 'article_heading', 'appendix_heading']:
                    level = 1

                if found_type == 'appendix_heading':
                    current_annex_context = raw_id 
                    parent_stack = [(0, "ROOT")] 

                # Stack Management
                while len(parent_stack) > 1 and parent_stack[-1][0] >= level:
                    parent_stack.pop()
                
                parent_id = parent_stack[-1][1]
                parent_stack.append((level, raw_id))
                
                current_chunk = {
                    'chunk_id': chunk_counter, 'clause_id': raw_id, 'parent_id': parent_id,
                    'level': level, 'appendix/annex': current_annex_context, 
                    'content_verbatim': line + "\n", 'source_page': str(page_num)
                }
            else:
                current_chunk['content_verbatim'] += line + " "

    if current_chunk['content_verbatim'].strip(): 
        all_chunks.append(current_chunk)

    cols = ['chunk_id', 'clause_id', 'parent_id', 'level', 'appendix/annex', 'content_verbatim', 'source_page']
    df = pd.DataFrame(all_chunks)
    for c in cols:
        if c not in df.columns: df[c] = None
            
    return df[cols], debug_images

def _parse_paddle_output(output, model_type):
    """
    Helper to standardize output from different Paddle models into:
    [{'label': str, 'bbox': [x1,y1,x2,y2], 'order': int/None}, ...]
    """
    normalized_blocks = []
    
    if not output: return []
    
    # --- V3 LOGIC (Structure + Layout Enrichment) ---
    if model_type == "v3":
        paddle_data = output[0].json.get('res', {})
        structure_blocks = paddle_data.get('parsing_res_list', [])
        layout_boxes = paddle_data.get('layout_det_res', {}).get('boxes', [])
        
        # Enrich Labels via IoU
        for s_block in structure_blocks:
            best_label = s_block.get('block_label', 'text')
            max_iou = 0.0
            s_bbox = s_block.get('block_bbox')
            if not s_bbox: continue
            
            for l_box in layout_boxes:
                iou = calculate_iou(s_bbox, l_box.get('coordinate'))
                if iou > max_iou:
                    max_iou = iou
                    best_label = l_box.get('label')
            
            if max_iou > 0.80:
                s_block['block_label'] = best_label
                
        # Format
        for blk in structure_blocks:
            normalized_blocks.append({
                'label': blk.get('block_label', 'text'),
                'bbox': blk.get('block_bbox'),
                'order': blk.get('block_order')
            })

    # --- VL LOGIC (Direct JSON Parse) ---
    elif model_type == "vl":
        # Handle list vs single object return
        res_list = list(output)
        if not res_list: return []
        
        page_res = res_list[0]
        data = page_res.json if hasattr(page_res, 'json') else page_res
        
        raw_blocks = []
        if 'res' in data and 'parsing_res_list' in data['res']:
            raw_blocks = data['res']['parsing_res_list']
        elif 'res' in data and 'layout_det_res' in data['res']:
            raw_blocks = data['res']['layout_det_res']['boxes']
            
        for blk in raw_blocks:
            # VL keys can vary slightly
            bbox = blk.get('block_bbox') or blk.get('coordinate') or blk.get('bbox')
            label = blk.get('block_label') or blk.get('label') or 'text'
            
            if bbox:
                normalized_blocks.append({
                    'label': label,
                    'bbox': bbox,
                    'order': None # VL often implies order by list position
                })
                
    return normalized_blocks

def hybrid_extract_page_data(pdf_path, page_number, model, model_type="v3", exclude_labels=None):
    if exclude_labels is None:
        exclude_labels = ['header', 'footer']

    print(f"--- Processing Page {page_number} (Model: {model_type.upper()}) ---")
    
    # 1. Open PDF & Prep Image
    doc = fitz.open(pdf_path)
    if page_number > len(doc): return pd.DataFrame(), None
    page = doc[page_number - 1] 
    
    # Determine Zoom based on model type
    zoom = 2.0 if model_type == "vl" else 1.5 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    
    img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    if pix.n == 4: img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2RGB)
    img_bgr = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)

    # 2. Predict
    layout_results = model.predict(img_bgr)
    
    # 3. Standardize Output
    all_blocks = _parse_paddle_output(layout_results, model_type)
    
    # 4. Filter & Sort
    valid_blocks = [b for b in all_blocks if b['label'].lower() not in exclude_labels]
    valid_blocks.sort(key=get_sorting_key)

    # 5. Calculate Scale
    image_h, image_w = img_bgr.shape[:2]
    scale_x = page.rect.width / image_w
    scale_y = page.rect.height / image_h

    # 6. Extraction Loop
    viz_image = img_bgr.copy()
    extracted_lines = []
    table_counter = 1

    print("\n" + "="*100)
    print(f"{'ORDER':<6} | {'LABEL':<12} | TEXT CONTENT")
    print("="*100)

    for idx, block in enumerate(valid_blocks):
        label = block['label'].lower()
        bbox = block['bbox']
        
        # Viz
        x1, y1, x2, y2 = map(int, bbox)
        color = (0, 0, 255) if 'table' in label else (0, 255, 0)
        cv2.rectangle(viz_image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(viz_image, f"[{idx+1}] {label.upper()}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Extract
        clean_text = ""
        if 'table' in label:
            clean_text = f"<Table_{table_counter}>"
            extracted_lines.append(clean_text)
            table_counter += 1
        else:
            rect = fitz.Rect(bbox[0] * scale_x, bbox[1] * scale_y, 
                             bbox[2] * scale_x, bbox[3] * scale_y)
            text = page.get_text("text", clip=rect).strip()
            if text:
                lines = text.split('\n')
                extracted_lines.extend([l.strip() for l in lines if l.strip()])
                clean_text = text.replace('\n', ' ')
        
        if clean_text:
             print(f"[{idx+1:<4}] | [{label.upper():<10}] : {clean_text[:75]}...")

    print("="*100 + "\n")

    # 7. Regex Hierarchy Parsing (Updated Logic)
    chunks = []
    parent_stack = [(0, "ROOT")] 
    
    # --- NEW: Persistent Context Variable ---
    current_annex_context = "NIL"
    
    current_chunk = {
        'chunk_id': 1, 'clause_id': "NIL", 'parent_id': "ROOT", 'level': 0,
        'appendix/annex': "NIL", 'content_verbatim': "", 'source_page': page_number
    }
    
    patterns = get_compliance_patterns()
    regex_map = {k: re.compile(v) for k, v in patterns.items()}
    heading_priority = [
        'appendix_heading', 'cfr_part_heading', 'cfr_section_heading', 
        'fmvss_paragraph', 'article_heading', 'multi_level_heading', 
        'roman_upper_heading', 'numeric_paren_heading', 'alpha_paren_heading'
    ]

    for line in extracted_lines:
        if line.startswith("<Table_"):
            current_chunk['content_verbatim'] += f"\n{line}\n"
            continue

        match_found = False; found_type = None; match_obj = None
        for h_type in heading_priority:
            m = regex_map[h_type].match(line)
            if m:
                match_found = True; found_type = h_type; match_obj = m
                break
        
        if match_found:
            # 1. Archive Previous Chunk
            if current_chunk['content_verbatim'].strip(): chunks.append(current_chunk)
            
            # 2. Extract ID (FIXED for Appendix)
            if found_type == 'appendix_heading':
                # Capture full ID: "APPENDIX A" instead of just "APPENDIX"
                raw_id = f"{match_obj.group(1)} {match_obj.group(2)}"
                # Update persistent context
                current_annex_context = raw_id
                # Reset stack for new Appendix
                parent_stack = [(0, "ROOT")] 
                level = 1
            else:
                raw_id = match_obj.group(1).strip().strip('.')
                # Standard level calculation
                level = raw_id.count('.') + 1
                if found_type in ['cfr_part_heading', 'article_heading']: level = 1

            # 3. Stack Update
            while len(parent_stack) > 1 and parent_stack[-1][0] >= level:
                parent_stack.pop()
            
            parent_id = parent_stack[-1][1]
            parent_stack.append((level, raw_id))
            
            # 4. Create New Chunk (Inherit persistent context)
            current_chunk = {
                'chunk_id': len(chunks) + 1, 
                'clause_id': raw_id, 
                'parent_id': parent_id,
                'level': level, 
                'appendix/annex': current_annex_context, # Uses persistent variable
                'content_verbatim': line + "\n", 
                'source_page': page_number
            }
        else:
            current_chunk['content_verbatim'] += line + " "

    if current_chunk['content_verbatim'].strip(): chunks.append(current_chunk)

    cols = ['chunk_id', 'clause_id', 'parent_id', 'level', 'appendix/annex', 'content_verbatim', 'source_page']
    df = pd.DataFrame(chunks)
    for c in cols:
        if c not in df.columns: df[c] = None
            
    return df[cols], viz_image

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # --- CONFIGURATION ---
    PDF_FILE = "AN_AP/gso-ece-117-2024-en.pdf"
    # TARGET_PAGE = 6
    PAGES_LIST = [12]
    
    # SWITCH HERE: "v3" (Original) or "vl" (New Vision-Language)
    SELECTED_MODEL = "vl" 
    
    # 1. Load Model
    model_pipeline = load_model(SELECTED_MODEL)
    
    # 2. Process
    # df_result, debug_img = hybrid_extract_page_data(
    #     PDF_FILE, TARGET_PAGE, 
    #     model_pipeline, 
    #     model_type=SELECTED_MODEL,
    #     exclude_labels=['header', 'footer','footnote','number']
    # )

    df_result, img_dict = hybrid_extract_data(
        PDF_FILE, PAGES_LIST, 
        model_pipeline, 
        model_type=SELECTED_MODEL,
        exclude_labels=['header', 'footer','footnote','number']
    )
    
    print("\n--- Extracted Data ---")
    page_to_view = PAGES_LIST[0] 
    
    if page_to_view in img_dict:
        plt.figure(figsize=(15, 20))
        # Convert BGR (OpenCV) to RGB (Matplotlib)
        plt.imshow(cv2.cvtColor(img_dict[page_to_view], cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Visual Debug - Page {page_to_view} ({SELECTED_MODEL.upper()} Model)")
        plt.show()
    else:
        print(f"No debug image found for Page {page_to_view}")

df_result

/home/ubuntu/miniconda3/envs/PaddleOCR_Instance/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PaddleOCR VL Model...


/home/ubuntu/miniconda3/envs/PaddleOCR_Instance/lib/python3.10/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/home/ubuntu/miniconda3/envs/PaddleOCR_Instance/lib/python3.10/site-packages/paddle/base/framework.py:829: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(
Creating model: ('PP-DocLayoutV2', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ubuntu/.paddlex/official_models/PP-DocLayoutV2`.
Creating model: ('PaddleOCR-VL-0.9B', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ubuntu/.paddlex/official_models/PaddleOCR-V

--- Processing 1 Pages (Model: VL) ---


FileNotFoundError: no such file: 'AN_AP/gso-ece-117-2024-en.pdf'